# Plantilla de algoritmo. Parte 3 (Problema del Viajante)

**Cómo usarla:** copia este archivo como `parte3/<algoritmo>_<tunombre>.ipynb` (por ejemplo, `parte3/fuerza_bruta_pablo.ipynb`) y trabaja solo **debajo de la línea ✂️**. Lo de arriba (preparación) es el mismo código de la Parte 2 del trabajo y sirve para cargar las versiones del grafo: **no lo modifiques**. La guía completa está en `CONTRIBUTING.md`.

> ⚙️ El notebook se ejecuta desde su carpeta (`parte3/`): la primera celda sube a la raíz del proyecto para que las rutas (`data/...`) funcionen.

In [1]:
# Sube a la raíz del proyecto para que las rutas (data/ciudades.csv, data/versiones/...) funcionen
import os
from pathlib import Path
if Path.cwd().name == "parte3":
    os.chdir("..")

In [2]:
# Librerías que se usan en todo el notebook
import csv                       # leer y escribir tablas en formato CSV
import itertools                 # recorrer todas las combinaciones de parámetros
import unicodedata               # quitar acentos de los nombres (Cataluña -> cataluna)
from pathlib import Path         # rutas de archivos

import networkx as nx            # crear y analizar grafos (la misma librería de las prácticas)
import numpy as np               # tablas de números (las tablas de distancias)
from scipy.sparse import csr_matrix                 # matriz del grafo para el algoritmo de Dijkstra
from scipy.sparse.csgraph import dijkstra           # caminos mínimos desde varios orígenes
from scipy.spatial import cKDTree                   # buscar rápido el nodo más cercano a un punto

from tsp_utils import dibujar_red  # función de dibujo de la red de carreteras, comentada en tsp_utils.py

In [3]:
# ── Parámetros de las versiones (editables) ──
X_KM  = 10             # una ciudad a más de X_KM km de la red se deja fuera de la versión
N_MIN = 4              # mínimo de ciudades para una versión por comunidad autónoma
RECALCULAR = False     # True: vuelve a calcular las 16 tablas (necesita el grafo G de la Parte 1)
CARPETA = Path("data/versiones")     # aquí se guardan las tablas ya calculadas

# Qué tipos de vía (`highway`) puede usar cada versión de carreteras
CARRETERAS = {
    "completo":             ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link", "secondary", "secondary_link"],
    "autovias_principales": ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link"],
    "solo_autovias":        ["motorway", "motorway_link"],
    "sin_autovias":         ["trunk", "trunk_link", "primary", "primary_link",
                             "secondary", "secondary_link"],
}
PESOS = {"km": ("length", 1000), "tiempo": ("travel_time", 1)}   # peso -> (atributo de la arista, divisor)
CONJUNTOS = ["capitales", "capitales_y_100k"]                    # qué ciudades se visitan
GENERALES = ["multigrafo_dirigido", "grafo_dirigido",
             "multigrafo_no_dirigido", "grafo_simple_no_dirigido"]

In [4]:
# Lee la lista de ciudades (nombre, comunidad, si es capital de provincia, coordenadas...)
with open("data/ciudades.csv", encoding="utf-8") as f:
    CIUDADES = list(csv.DictReader(f))
for c in CIUDADES:
    c["lat"], c["lon"] = float(c["lat"]), float(c["lon"])
    c["capital"] = c["capital_provincia"] == "True"

def slug(texto):
    # "Castilla y León" -> "castilla_y_leon" (sin acentos ni espacios, para usarlo como nombre)
    sin_acentos = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return sin_acentos.lower().replace(" ", "_").replace("-", "_")

AMBITOS = ["espana"] + sorted({slug(c["comunidad"]) for c in CIUDADES})

def ciudades_de(nodos, ambito):
    # Nombres de las ciudades de un conjunto: nodos = "capitales" o "capitales_y_100k";
    # ambito = "espana" o una comunidad (p. ej. "galicia")
    return [c["nombre"] for c in CIUDADES
            if (c["capital"] or nodos == "capitales_y_100k")
            and (ambito == "espana" or slug(c["comunidad"]) == ambito)]

print(f"{len(CIUDADES)} ciudades | {len(ciudades_de('capitales', 'espana'))} capitales | ámbitos: {len(AMBITOS) - 1} comunidades")
print("Capitales de Galicia:", ciudades_de("capitales", "galicia"))

75 ciudades | 47 capitales | ámbitos: 15 comunidades
Capitales de Galicia: ['A Coruña', 'Ourense', 'Lugo', 'Pontevedra']


In [5]:
def nombre_tabla(aristas, peso, dirigido):
    return CARPETA / f"{aristas}__{peso}__{'dirigido' if dirigido else 'no_dirigido'}.csv"

def guardar_tabla(D, nombres, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        np.savetxt(f, D, delimiter=",", header=",".join(nombres), comments="", fmt="%.3f")

def leer_tabla(aristas, peso, dirigido):
    ruta = nombre_tabla(aristas, peso, dirigido)
    with open(ruta, encoding="utf-8") as f:
        nombres = f.readline().strip().split(",")        # primera fila: nombres de las ciudades
    return np.loadtxt(ruta, delimiter=",", skiprows=1, encoding="utf-8"), nombres

def escribir_csv(ruta, filas, campos):
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        w.writeheader()
        w.writerows(filas)

In [6]:
UNIDAD = {"km": "km", "tiempo": "segundos"}

def cargar_version(aristas, nodos, ambito, peso, general):
    # aristas: "completo" | "autovias_principales" | "solo_autovias" | "sin_autovias"
    # nodos:   "capitales" | "capitales_y_100k"       ambito: "espana" | "galicia" | "andalucia" ...
    # peso:    "km" | "tiempo"                        general: uno de GENERALES
    # (multigrafo y grafo simple dan el mismo grafo de ciudades: no hay aristas repetidas)
    dirigido = "no_dirigido" not in general
    D, nombres = leer_tabla(aristas, peso, dirigido)
    elegidas = [c for c in ciudades_de(nodos, ambito) if c in nombres]    # sin las que quedaron fuera
    if ambito != "espana" and len(elegidas) < N_MIN:
        raise ValueError(f"Solo hay {len(elegidas)} ciudades (mínimo {N_MIN}) en {ambito}")
    K = nx.DiGraph() if dirigido else nx.Graph()
    for c in CIUDADES:
        if c["nombre"] in elegidas:
            K.add_node(c["nombre"], lat=c["lat"], lon=c["lon"])
    for p, a in enumerate(elegidas):
        for q, b in enumerate(elegidas):
            if p != q and (dirigido or p < q):                            # no dirigido: una arista por par
                K.add_edge(a, b, peso=float(D[nombres.index(a), nombres.index(b)]))
    K.graph.update(aristas=aristas, nodos=nodos, ambito=ambito, peso=peso,
                   unidad=UNIDAD[peso], general=general,
                   fuera=[c for c in ciudades_de(nodos, ambito) if c not in nombres])
    return K

In [7]:
# [AYUDA COMPARTIDA] Funciones comunes a todos los algoritmos
import time
import tracemalloc

def coste_ruta(K, ruta):
    # Coste total del ciclo: suma de las distancias entre ciudades consecutivas y de la última a la primera
    return sum(K[ruta[i]][ruta[(i + 1) % len(ruta)]]["peso"] for i in range(len(ruta)))

def validar_ruta(K, ruta):
    # Comprueba que la ruta visita cada ciudad de K exactamente una vez
    assert len(ruta) == K.number_of_nodes() and set(ruta) == set(K.nodes), "la ruta no visita cada ciudad una vez"

def medir(algoritmo, K, memoria=False, **opciones):
    # Ejecuta algoritmo(K, **opciones), comprueba el resultado y devuelve cuánto ha tardado
    # (con memoria=True mide también el pico de memoria en una segunda ejecución, porque medirla ralentiza)
    t0 = time.perf_counter()
    ruta, coste = algoritmo(K, **opciones)
    segundos = time.perf_counter() - t0
    validar_ruta(K, ruta)
    assert abs(coste - coste_ruta(K, ruta)) < 1e-6 * max(1.0, abs(coste)), "el coste devuelto no coincide con el de la ruta"
    resultado = {"algoritmo": algoritmo.__name__, "ciudades": K.number_of_nodes(), "coste": round(coste, 3),
                 "unidad": K.graph.get("unidad"), "segundos": round(segundos, 4)}
    if memoria:
        tracemalloc.start()
        algoritmo(K, **opciones)
        _, pico = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        resultado["memoria_mb"] = round(pico / 1e6, 2)
    return resultado

---
### ✂️ A PARTIR DE AQUÍ SE INTEGRA EN EL TRABAJO

Todo lo que escribas **debajo de esta línea** pasará al notebook principal (`trabajo_tsp.ipynb`, Parte 3). No modifiques esta celda.

### Fuerza bruta (Pablo Rivas)

<div style="border-left:5px solid #4C72B0;background:#EEF2FA;color:#000;padding:10px 14px;margin:10px 0;border-radius:4px;">

**📚 TEORÍA**

La estrategia más directa que existe: como el viajante tiene que pasar por *todas* las ciudades una sola vez y volver al principio, basta con **probar todos los órdenes posibles**, calcular cuánto se recorre en cada uno y quedarse con el más barato. No hay atajos, ni adivinanzas, ni podas: se paga por completo el coste de la enumeración, y a cambio se obtiene una garantía absoluta de que la solución encontrada **es** la óptima.

**Aplicado al problema del viajante.** Dado el grafo de ciudades `K` (completo: todas las ciudades conectadas con todas), un *ciclo hamiltoniano* es un orden `c₁, c₂, …, cₙ` donde cada ciudad aparece exactamente una vez y, al terminar, se vuelve de `cₙ` a `c₁`. Su coste es la suma de las aristas que unen ciudades consecutivas, más la de la vuelta. El TSP consiste en encontrar el ciclo hamiltoniano de coste mínimo, y este algoritmo lo resuelve listando **todos** los ciclos posibles y evaluándolos uno a uno.

**El truco de la simetría del ciclo.** Un ciclo de `n` ciudades admite `n!` formas de escribirse, pero son el mismo recorrido: `A→B→C→A`, `B→C→A→B` y `C→A→B→C` recorren exactamente el mismo camino y cuestan lo mismo. Luego basta **fijar la primera ciudad** (cualquiera sirve: el óptimo no cambia) y permutar únicamente las otras `n−1`. Se pasa de `n!` ciclos a `(n−1)!`, es decir, un ahorro de un factor `n` sin dejar fuera ni un candidato. El parámetro `inicio` de la función fija esa ciudad (por defecto, la primera de `K.nodes`); en un grafo **dirigido** la ciudad fijada sí importa, porque el ciclo `A→B→C→A` puede costar distinto que `B→A→C→B`.

**Pseudocódigo.**

<pre style="background:#fff;border:1px solid #ccd;padding:10px 14px;border-radius:4px;font-size:13px;">
FUERZA_BRUTA(K, inicio):
    si K no tiene ciudades: devolver ([], 0)
    inicio  ← la ciudad dada, o la primera de K si no se dio ninguna
    resto   ← todas las ciudades de K menos inicio
    mejor_ruta ← nada ;  mejor_coste ← infinito

    para cada permutación p de resto:                 # (n-1)! permutaciones
        ruta  ← [inicio] + p
        coste ← 0 ;  anterior ← inicio
        para cada ciudad de p:                          # suma incremental, arista a arista
            coste ← coste + K[anterior][ciudad]["peso"]
            anterior ← ciudad
        coste ← coste + K[anterior][inicio]["peso"]     # la vuelta, que cierra el ciclo

        si coste es menor que mejor_coste:
            mejor_ruta ← ruta ;  mejor_coste ← coste

    devolver (mejor_ruta, mejor_coste)
</pre>

**Complejidad.** **Tiempo: O((n−1)!·n)** — hay `(n−1)!` permutaciones y cada una exige `O(n)` operaciones para sumar sus aristas. **Memoria: O(n)** — solo se guardan la mejor ruta y la permutación en curso; los ciclos descartados no se almacenan jamás (se mide abajo: ~0,6 KB, constante).

**Propiedades.**

- **Tipo:** **exacto** (enumeración completa del espacio de soluciones).
- **¿Siempre encuentra el óptimo?** **Sí**, por construcción: evalúa todos los ciclos posibles. Es la referencia contra la que se compara el resto de algoritmos.
- **¿Determinista?** **Sí**: no usa azar, así que con la misma entrada devuelve siempre la misma ruta.
- **¿Garantía de calidad?** 100 % (es el óptimo), pero a un precio computacional que se dispara muy pronto.
- **¿Necesita distancias simétricas?** **No.** No supone que `peso(a,b) = peso(b,a)`, de modo que funciona igual con `nx.DiGraph` (donde, por los sentidos únicos, Madrid→Sevilla y Sevilla→Madrid difieren) y con `nx.Graph`.
- **¿Hasta cuántas ciudades es razonable?** Cómodo hasta **10 ciudades** (≈0,5 s medidos). En 11 se van ~6 s y en 12 ~68 s: como el tiempo crece con el factorial, cada ciudad añadida multiplica el coste por `n`. **Más de 12 ciudades es inviable** en la práctica; ahí hacen falta algoritmos mejores (Held–Karp, vecino más cercano, 2-opt…).

</div>

**💻 CÓDIGO**

La implementación recorre las `(n−1)!` permutaciones con `itertools.permutations` y **suma el coste de forma incremental** (arista a arista, arrancando desde la ciudad anterior) en vez de recalcular la ruta entera con cada candidato: mismo resultado, pero un factor `n` más rápido. No se usa poda (*branch & bound*), para que siga siendo fuerza bruta «puro» y sea comparable con los demás algoritmos de la Parte 3.

In [8]:
def fuerza_bruta(K, inicio=None):
    # Fuerza bruta para el problema del viajante: prueba todos los ciclos posibles y se queda con
    # el más barato. Contrato: K es el grafo de ciudades (completo, K[a][b]["peso"] = distancia de
    # a a b) y devuelve (ruta, coste), donde `ruta` es la lista con todas las ciudades, cada una una
    # vez, en el orden en que se visitan (el viajante vuelve después a la primera), y `coste` es el
    # coste total del ciclo, incluida esa vuelta. No asume simetría: sirve para K dirigido (nx.DiGraph)
    # y no dirigido (nx.Graph) por igual.

    ciudades = list(K.nodes)

    # Casos triviales: sin ciudades o con una sola, el ciclo ya está determinado y no hay nada que probar
    if not ciudades:
        return [], 0.0
    if inicio is None:
        inicio = ciudades[0]                       # por defecto, la primera ciudad del grafo
    elif inicio not in K:
        raise KeyError(f"La ciudad de inicio {inicio!r} no está en el grafo")
    resto = [c for c in ciudades if c != inicio]
    if not resto:
        return [inicio], 0.0

    # Truco de la simetría del ciclo: fijamos la primera ciudad y solo permutamos las otras n-1.
    # Así bajamos de (n!) ciclos a ((n-1)!) sin dejar de probar ni un solo ciclo distinto.
    mejor_ruta = None
    mejor_coste = float("inf")

    for p in itertools.permutations(resto):
        ruta = (inicio,) + p

        # Sumamos el coste arista a arista (acumulando desde la ciudad anterior) en lugar de
        # recalcular la ruta entera: sale el mismo número, pero cuesta O(n) por ciclo, no O(n^2)
        coste = 0.0
        anterior = inicio
        for ciudad in p:
            coste += K[anterior][ciudad]["peso"]
            anterior = ciudad
        coste += K[anterior][inicio]["peso"]        # la vuelta a la ciudad de inicio, que cierra el ciclo

        # Nos quedamos solo con el mejor ciclo de los probados; los demás se descartan y no se guardan,
        # de ahí que la memoria sea O(n) y no O((n-1)!·n)
        if coste < mejor_coste:
            mejor_ruta, mejor_coste = ruta, coste

    return list(mejor_ruta), mejor_coste

**Prueba rápida.** Tres comprobaciones, de menor a mayor dificultad:

1. **Versión pequeña real:** las 6 ciudades de Galicia, empezando en Vigo.
2. **¿De verdad encuentra el óptimo?** Un subgrafo de 5 ciudades de Galicia, comparando contra una referencia independiente que evalúa **los `n!` ciclos** con `coste_ruta` (sin el truco de fijar ciudad). Si ambas coinciden, fijar la primera ciudad no ha descartado ningún candidato bueno.
3. **¿Hasta cuánto aguanta?** Escalabilidad sobre Andalucía (13 ciudades): se van añadiendo ciudades y se miden ciclos, tiempo y memoria.

In [9]:
# ── 1. Versión pequeña real: Galicia (6 ciudades), empezando en Vigo ──
K = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="galicia", peso="km", general="grafo_dirigido")
ruta, coste = fuerza_bruta(K, inicio="Vigo")
validar_ruta(K, ruta)                              # comprueba que la ruta visita cada ciudad una vez
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print(medir(fuerza_bruta, K, memoria=True, inicio="Vigo"))

# ── 2. Comprobación de optimalidad contra una referencia independiente ──
# La referencia evalúa los n! ciclos completos con coste_ruta, sin fijar ninguna ciudad: es lenta,
# pero sirve de "verdad" para asegurar que fijar la primera ciudad no ha descartado el óptimo.
K5 = K.subgraph(sorted(K.nodes)[:5]).copy()         # subgrafo completo de 5 ciudades de Galicia
n_ciclos = 0
referencia = float("inf")
for p in itertools.permutations(K5.nodes):          # los 5! = 120 ciclos posibles, sin ningún atajo
    n_ciclos += 1
    referencia = min(referencia, coste_ruta(K5, list(p)))
ruta5, coste5 = fuerza_bruta(K5)
validar_ruta(K5, ruta5)
assert abs(coste5 - referencia) < 1e-9, "fuerza_bruta no coincide con la enumeración completa"
ciclos_fb = 1                                        # (n-1)! = 4·3·2 = 24 ciclos que prueba el algoritmo
for i in range(2, 5):
    ciclos_fb *= i
print(f"\nGalicia-5 → referencia: {referencia:.3f} km ({n_ciclos} ciclos con coste_ruta) | "
      f"fuerza_bruta: {coste5:.3f} km (solo {ciclos_fb} ciclos) | óptimo confirmado ✔")

Vigo → Ourense → Lugo → A Coruña → Santiago de Compostela → Pontevedra | coste: 460.4 km
{'algoritmo': 'fuerza_bruta', 'ciudades': 6, 'coste': 460.438, 'unidad': 'km', 'segundos': 0.0001, 'memoria_mb': 0.0}

Galicia-5 → referencia: 430.750 km (120 ciclos con coste_ruta) | fuerza_bruta: 430.750 km (solo 24 ciclos) | óptimo confirmado ✔


In [10]:
# ── 3. Escalabilidad: cuántas ciudades aguanta antes de volverse inviable ──
# Cargamos Andalucía (13 ciudades) y vamos cogiendo subgrafos más grandes. Como el tiempo crece con
# el factorial, nos paramos solos al superar ~30 s para no bloquear el notebook.
KAnd = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="andalucia", peso="km", general="grafo_dirigido")
ciudades = sorted(KAnd.nodes)
LIMITE_S = 30

print(f"Andalucía: {KAnd.number_of_nodes()} ciudades disponibles")
print(f"{'ciudades':>9} | {'ciclos (n-1)!':>14} | {'segundos':>9} | {'coste (km)':>10}")
for n in range(6, KAnd.number_of_nodes() + 1):
    Kn = KAnd.subgraph(ciudades[:n]).copy()         # subgrafo completo de las n primeras ciudades
    t0 = time.perf_counter()
    ruta_n, coste_n = fuerza_bruta(Kn)
    segundos = time.perf_counter() - t0
    validar_ruta(Kn, ruta_n)
    ciclos = 1
    for i in range(2, n):                           # (n-1)! permutaciones realmente probadas
        ciclos *= i
    print(f"{n:>9} | {ciclos:>14,} | {segundos:>9.3f} | {coste_n:>10.1f}")
    if segundos > LIMITE_S:
        print(f"   ↑ nos paramos aquí: ya supera {LIMITE_S} s, y cada ciudad más multiplica el tiempo por n")
        break

# La memoria es O(n): apenas cambia al añadir ciudades, porque solo se guardan la mejor ruta y la
# permutación en curso (los ciclos descartados no se almacenan). El subgrafo se crea FUERA de la
# medición, para que el pico que reporte tracemalloc sea el del algoritmo y no el del propio grafo.
print("\nMemoria pico del algoritmo (el subgrafo ya está creado):")
for n in (6, 8, 10, 11):
    Ksub = KAnd.subgraph(ciudades[:n]).copy()
    tracemalloc.start()
    fuerza_bruta(Ksub)
    _, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    print(f"  {n:>2} ciudades -> {pico / 1e3:>7.1f} KB")

Andalucía: 13 ciudades disponibles
 ciudades |  ciclos (n-1)! |  segundos | coste (km)
        6 |            120 |     0.000 |     1006.4
        7 |            720 |     0.001 |     1197.6
        8 |          5,040 |     0.006 |     1231.4
        9 |         40,320 |     0.053 |     1235.1


       10 |        362,880 |     0.530 |     1237.7


       11 |      3,628,800 |     5.764 |     1241.0


       12 |     39,916,800 |    69.176 |     1251.3
   ↑ nos paramos aquí: ya supera 30 s, y cada ciudad más multiplica el tiempo por n

Memoria pico del algoritmo (el subgrafo ya está creado):
   6 ciudades ->     0.5 KB
   8 ciudades ->     1.3 KB


  10 ciudades ->     4.7 KB


  11 ciudades ->     4.1 KB


<div style="border-left:5px solid #55A868;background:#EAF6ED;color:#000;padding:10px 14px;margin:10px 0;border-radius:4px;">

**🧮 RESULTADOS**

**Corrección.** En el subgrafo de 5 ciudades de Galicia, la referencia que evalúa los `5! = 120` ciclos con `coste_ruta` devuelve **exactamente el mismo coste** (430,750 km) que `fuerza_bruta`, que solo prueba `(5−1)! = 24` ciclos: fijar la primera ciudad no descarta ningún candidato bueno. `validar_ruta` confirma además en todos los casos que la ruta visita cada ciudad exactamente una vez, y la comprobación de `medir` que el coste devuelto coincide con el de la ruta devuelta.

**Óptimo de las 6 ciudades de Galicia** (versión `completo` / `capitales_y_100k` / `km` / `grafo_dirigido`), empezando en Vigo:

> **Vigo → Ourense → Lugo → A Coruña → Santiago de Compostela → Pontevedra → Vigo = 460,4 km**

**Escalabilidad** (mismo peso y tipo de grafo, con `ambito="andalucia"`; cada fila usa un subgrafo de las `n` primeras ciudades ordenadas alfabéticamente):

| Ciudades | Ciclos probados `(n−1)!` | Coste | Tiempo | Memoria pico |
|---:|---:|---:|---:|---:|
| 6 | 120 | 1.006,4 km | 0,000 s | 0,5 KB |
| 8 | 5.040 | 1.231,4 km | 0,006 s | 1,3 KB |
| 10 | 362.880 | 1.237,7 km | 0,53 s | 4,7 KB |
| 11 | 3.628.800 | 1.241,0 km | 5,76 s | 4,1 KB |
| 12 | 39.916.800 | 1.251,3 km | 69,2 s | — |

**Conclusiones.**

- El algoritmo es **correcto y óptimo por definición**: devuelve siempre el ciclo de coste mínimo, y se ha verificado contra una enumeración independiente de todos los ciclos posibles.
- **El cuello de botella es el tiempo, no la memoria.** La memoria se mueve en el rango de **unos pocos KB** (0,5 → 4,7 KB de 6 a 11 ciudades, un crecimiento muy suave acorde con `O(n)`), mientras que el tiempo pasa de 0,000 s a 69 s en el mismo intervalo.
- **Funciona bien hasta 10 ciudades** (medio segundo). En 11 ya son ~6 s y en 12 ~70 s: la escala la marca el **factorial**, no el número de ciudades ni el tamaño del grafo.
- Por eso, **hasta ~10–11 ciudades es la opción recomendable**. A partir de ahí el crecimiento `O((n−1)!·n)` lo hace inutilizable (13 ciudades serían del orden de la hora y media; 20, millones de años). Para las 47 capitales o las 75 ciudades hace falta **Held–Karp** (exacto hasta ~20 ciudades) o **heurísticas** (vecino más cercano, 2-opt, Christofides, genético), que son el resto de la Parte 3.
- Su principal valor es el de **línea base**: cualquier otro algoritmo puede compararse con este óptimo en las versiones pequeñas para medir cuánto se aleja de la mejor solución posible.

</div>

---